# Feature Engineering - ATP Tennis Match Predictor

This notebook builds the feature set used to predict ATP match winners based on findings from the EDA notebook (`01_eda.ipynb`). Rather than using raw match data directly, this notebook transforms it into features that describe the *relative* gap between two players: their ranking, points, and historical performance, since that's what actually determines who's more likely to win a given match.

**Key decisions carried over from EDA:**
- `Pts_1`/`Pts_2` use `-1` as a missing data placeholder in ~23% of rows (not real NaN)
- `Odd_1`/`Odd_2` are only reliably available from ~2005 onward
- `rank_diff` (corr = -0.24) and `points_diff` (corr = 0.32) both showed real relationships with match outcome and are treated as core features

CSVs don't preserve datetime types, so 'Date' needs to be re-converted with 'pd.to_datetime()' every time the file is reloaded.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/atp_matches_clean.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.shape

(68274, 21)

`points_data_missing` is a separate binary column (1 = we don't know the points gap) that the model can learn to use as a signal, letting it learn to trust 'points_diff' less whenever this flag is on.

In [2]:
df['points_data_missing'] = ((df['Pts_1'] == -1) | (df['Pts_2'] == -1)).astype(int)

df['points_diff'] = np.where(
    df['points_data_missing'] == 1,
    0,
    df['Pts_1'] - df['Pts_2']
)

df['points_data_missing'].mean()

np.float64(0.22888654539063186)

`has_odds` is kept as a flag so any odds-based feature can later be built and evaluated only on the subset of rows where it's available

In [3]:
df['has_odds'] = ((df['Odd_1'] != -1) & (df['Odd_2'] != -1)).astype(int)
df['has_odds'].mean()

np.float64(0.9448252629112107)

The features in this notebook depends on chronological order. `reset_index(drop=True)` renumbers rows after sorting

In [4]:
df = df.sort_values('Date').reset_index(drop=True)

### Why reshape the data?

Each row in `df` is one match with two players side by side (`Player_1` and `Player_2` as separate columns). To calculate something like "Federer's win rate going into this match," it's easier if each player's participation in a match is its own row, rather than being split across two columns depending on which side of the match they happened to be listed on.

In [5]:
df = df.reset_index(drop=True)
df['match_id'] = df.index

p1 = df[['match_id', 'Date', 'Player_1', 'player_1_won', 'Surface']].copy()
p1.columns = ['match_id', 'Date', 'Player', 'Won', 'Surface']
p1['Slot'] = 'P1'

p2 = df[['match_id', 'Date', 'Player_2', 'player_1_won', 'Surface']].copy()
p2['Won'] = 1 - p2['player_1_won']
p2 = p2[['match_id', 'Date', 'Player_2', 'Won', 'Surface']]
p2.columns = ['match_id', 'Date', 'Player', 'Won', 'Surface']
p2['Slot'] = 'P2'

player_matches = pd.concat([p1, p2]).sort_values('Date').reset_index(drop=True)
print(df.shape[0] * 2 == player_matches.shape[0])
print(player_matches.duplicated(subset=['match_id', 'Player']).sum())
player_matches.head()

True
0


,match_id,Date,Player,Won,Surface,Slot
0,0,2000-01-03,Arthurs W.,0,Hard,P1
1,67,2000-01-03,Ilie A.,0,Hard,P2
2,66,2000-01-03,Fromberg R.,0,Hard,P2
3,65,2000-01-03,Henman T.,1,Hard,P2
4,64,2000-01-03,Henman T.,1,Hard,P2


### Leakage-safe rolling statistics

If a player's win rate for a given match accidentally includes the result of that same match, or a match that hasn't happened yet, the model is effectively being shown the answer before making its prediction.

In [6]:
player_matches['career_win_rate'] = (
    player_matches.groupby('Player')['Won']
    .transform(lambda x: x.shift().expanding().mean())
)

player_matches['surface_win_rate'] = (
    player_matches.groupby(['Player', 'Surface'])['Won']
    .transform(lambda x: x.shift().expanding().mean())
)

player_matches['recent_form'] = (
    player_matches.groupby('Player')['Won']
    .transform(lambda x: x.shift().rolling(window=10, min_periods=1).mean())
)

player_matches[['Player', 'Date', 'Won', 'career_win_rate', 'surface_win_rate', 'recent_form']].tail()

,Player,Date,Won,career_win_rate,surface_win_rate,recent_form
136543,Rublev A.,2026-07-18,1,0.630931,0.662162,0.6
136544,Collignon R.,2026-07-19,0,0.513514,0.615385,0.6
136545,Rublev A.,2026-07-19,1,0.631579,0.664430,0.6
136546,Darderi L.,2026-07-19,0,0.545455,0.693182,0.6
136547,Tsitsipas S.,2026-07-19,1,0.650000,0.728916,0.6


### Debugging note: merge row-count mismatch

Initial merge on [`Date`, `Player`] caused row count to increase from 68,274 to 125,629. Investigation showed the raw data has zero true duplicate matches, the issue was that players can legitimately play multiple real matches on the same calendar date (compressed tournament draws, and Masters Cup round-robin groups against different opponents). Adding `Round` to the merge key fixed most cases but still failed on round-robin groups, since 'Round' is the same label ("Round Robin") for multiple distinct matches in a group stage.

Fixed by assigning each match row a unique `match_id` before reshaping, and merging on that instead of trying to find a naturally unique combination of columns.

In [7]:
p1_stats = player_matches[player_matches['Slot'] == 'P1'].rename(columns={
    'career_win_rate': 'p1_career_win_rate',
    'surface_win_rate': 'p1_surface_win_rate',
    'recent_form': 'p1_recent_form'
})[['match_id', 'p1_career_win_rate', 'p1_surface_win_rate', 'p1_recent_form']]

p2_stats = player_matches[player_matches['Slot'] == 'P2'].rename(columns={
    'career_win_rate': 'p2_career_win_rate',
    'surface_win_rate': 'p2_surface_win_rate',
    'recent_form': 'p2_recent_form'
})[['match_id', 'p2_career_win_rate', 'p2_surface_win_rate', 'p2_recent_form']]

rows_before = df.shape[0]
df = df.merge(p1_stats, on=['match_id'], how='left')
df = df.merge(p2_stats, on=['match_id'], how='left')
print(rows_before == df.shape[0])

True


### Clean up helper columns

`match_id` and `Slot` were only needed to make the merge safe, they don't carry any predictive meaning themselves, so they're dropped.

In [8]:
df = df.drop(columns=['match_id'])
df.shape

(68274, 29)

### Handling players with no prior history

A brand-new player (their very first recorded match) has nothing to calculate a win rate from yet, so `p1_career_win_rate`, `p2_surface_win_rate`, etc. come back as `NaN` for them after the merge. Filling with `0.5` treats them as a neutral, unknown player (a 50/50 assumption), rather than `0` , which would wrongly treat every debut player as a guaranteed loser before they've even played a match.

In [9]:
history_cols = ['p1_career_win_rate', 'p2_career_win_rate',
                'p1_surface_win_rate', 'p2_surface_win_rate',
                'p1_recent_form', 'p2_recent_form']

for col in history_cols:
    df[col] = df[col].fillna(0.5)

df[history_cols].isnull().sum()

p1_career_win_rate     0
p2_career_win_rate     0
p1_surface_win_rate    0
p2_surface_win_rate    0
p1_recent_form         0
p2_recent_form         0
dtype: int64

### Building the final relative (difference) features

Every model-facing feature here is a difference, not a raw value tied to "Player 1" or "Player 2" as an arbitrary label. Player 1 vs Player 2 assignment in the raw data isn't meaningful, it's just how the match happened to be recorded, so giving the model raw absolute stats per slot risks it learning a false shortcut like "Player 1 tends to win more," which would be a data artifact, not a real pattern.

In [10]:
df['career_win_rate_diff'] = df['p1_career_win_rate'] - df['p2_career_win_rate']
df['surface_win_rate_diff'] = df['p1_surface_win_rate'] - df['p2_surface_win_rate']
df['recent_form_diff'] = df['p1_recent_form'] - df['p2_recent_form']

df[['career_win_rate_diff', 'surface_win_rate_diff', 'recent_form_diff']].describe()

,career_win_rate_diff,surface_win_rate_diff,recent_form_diff
count,68274.000000,68274.000000,68274.000000
mean,0.000236,0.001023,0.000479
std,0.189198,0.224188,0.247372
min,-1.000000,-1.000000,-1.000000
25%,-0.115877,-0.133333,-0.200000
50%,0.000000,0.000000,0.000000
75%,0.115954,0.133868,0.200000
max,1.000000,1.000000,1.000000


Worth remembering going into modelling: `rank_diff` and `points_diff` point in opposite directions for the same underlying idea. A negative `rank_diff` favours Player 1 (lower rank number = better), while a postive `points_diff` favours Player 1 (more points = better). The model doesn't care about sign consistency across features, but it matters for interpreting results later.

### Encoding Surface

Most models need numeric input, so the categorical `Surface` column (Hard/Clay/Grass/Carpet) is converted into separate binary (0/1) columns, one per category, with the first category dropped to avoid redundant information (if a match isn't Clay, Grass, or Hard, it must be Carpet).

In [11]:
df = pd.get_dummies(df, columns=['Surface'], drop_first=True)
df.filter(like='Surface_').columns.tolist()

['Surface_Clay', 'Surface_Grass', 'Surface_Hard']

### Sanity-checking the feature set

Before moving to modelling, every engineered feature should show some relationship with `player_1_won`. A feature with a correlation near zero isn't necessarily wrong to keep, but it's worth a note here about why it might still matter (e.g. it could interact with another feature even alone it shows little effect), or whether it's worth reconsidering.

In [12]:
feature_cols = ['rank_diff', 'points_diff', 'points_data_missing',
                'career_win_rate_diff', 'surface_win_rate_diff', 'recent_form_diff']

df[feature_cols + ['player_1_won']].describe()

,rank_diff,points_diff,points_data_missing,career_win_rate_diff,surface_win_rate_diff,recent_form_diff,player_1_won
count,68274.000000,68274.000000,68274.000000,68274.000000,68274.000000,68274.000000,68274.000000
mean,0.429900,-7.309283,0.228887,0.000236,0.001023,0.000479,0.500015
std,137.367839,2113.018457,0.420119,0.189198,0.224188,0.247372,0.500004
min,-4911.000000,-16641.000000,0.000000,-1.000000,-1.000000,-1.000000,0.000000
25%,-41.000000,-385.000000,0.000000,-0.115877,-0.133333,-0.200000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,41.000000,375.000000,0.000000,0.115954,0.133868,0.200000,1.000000
max,3381.000000,16516.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [13]:
df[feature_cols + ['player_1_won']].corr()['player_1_won'].sort_values()

rank_diff               -0.240164
points_data_missing      0.000089
surface_win_rate_diff    0.257014
recent_form_diff         0.267796
points_diff              0.283237
career_win_rate_diff     0.296353
player_1_won             1.000000
Name: player_1_won, dtype: float64

### Saving the engineered dataset

In [14]:
df.to_csv('../data/processed/atp_matches_features.csv', index=False)
df.shape

(68274, 34)

## Summary

Feature engineering was carried out based on domain knowledge and correlations
confirmed during EDA — `rank_diff`, `points_diff`, and rolling per-player
history features (career win rate, surface win rate, recent form), with
careful handling of two known data issues: disguised missing values in the
points columns, and structurally missing odds data before ~2005.

**Debugging note:** an early version of the player-history merge caused row
count to rise from 68,274 to 125,629. Investigation showed the raw data has
zero true duplicate matches. The cause was that players can legitimately play
multiple real matches on the same calendar date (compressed tournament draws,
and Masters Cup round-robin groups against different opponents on the same
day). Fixed by assigning each match a unique `match_id` before reshaping and
merging on that directly, rather than relying on a combination of real-world
columns (Date, Round, Player) to be unique.

Head-to-head record was deliberately left out of this first pass - a good
candidate to revisit after a baseline model is trained, once feature
importance and error analysis can show whether the model is actually missing
that kind of signal.

Next: `03_modeling.ipynb` — baseline model, then increasing complexity
(Logistic Regression → Random Forest → XGBoost), evaluated against a
time-based train/test split.